# EDA 11 — Loyers d'Annonce par Arrondissement (Koumoul / Data Fair)
**Source** : Koumoul opendata — Indicateurs de loyers d'annonce par commune 2025  
**Bronze path** : `data/bronze/rentals/date=YYYY-MM-DD/part-0.parquet`  
**Scope** : 20 arrondissements parisiens, appartements

## Schéma Bronze
| Colonne | Type | Description |
|---|---|---|
| `code_commune` | str | Code INSEE (751xx) |
| `arrondissement` | int | 1-20 |
| `loyer_m2` | float | Loyer médian prédit (€/m²/mois) |
| `loyer_m2_bas` | float | Borne basse (€/m²/mois) |
| `loyer_m2_haut` | float | Borne haute (€/m²/mois) |
| `nb_obs` | int | Nombre d'observations |
| `libgeo` | str | Libellé commune |
| `type_bien` | str | Appartement |


In [ ]:
import os, glob
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

pd.set_option("display.max_columns", None)
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

BRONZE_ROOT = os.path.abspath("../../data/bronze")
rental_files = sorted(glob.glob(f"{BRONZE_ROOT}/rentals/**/*.parquet", recursive=True))
print(f"Fichiers loyers trouvés : {len(rental_files)}")


In [ ]:
if rental_files:
    df = pd.concat([pd.read_parquet(f) for f in rental_files], ignore_index=True)
else:
    rng = np.random.default_rng(42)
    # Gradient réaliste de loyers Paris 2025
    base_loyers = {1:35,2:33,3:32,4:36,5:34,6:40,7:42,8:43,9:32,10:29,
                   11:30,12:27,13:26,14:29,15:31,16:39,17:34,18:26,19:22,20:23}
    df = pd.DataFrame([{
        "code_commune":   f"751{arr:02d}",
        "arrondissement": arr,
        "loyer_m2":       base_loyers[arr] + rng.normal(0, 1.5),
        "loyer_m2_bas":   base_loyers[arr] - 3 + rng.normal(0, 0.5),
        "loyer_m2_haut":  base_loyers[arr] + 3 + rng.normal(0, 0.5),
        "nb_obs":         int(rng.uniform(200, 1500)),
        "libgeo":         f"Paris {arr}e Arrondissement",
        "type_bien":      "Appartement",
        "ingested_at":    pd.Timestamp("now", tz="UTC"),
    } for arr in range(1, 21)])
    print("⚠️  Fichiers Bronze absents — démonstration sur données synthétiques")

print(f"Shape : {df.shape}")
df.head(5)


In [ ]:
# ── Loyers par arrondissement (avec intervalles de confiance) ─────────────────
df_sorted = df.sort_values("loyer_m2", ascending=False).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(14, 7))
x = range(len(df_sorted))
ax.bar(x, df_sorted["loyer_m2"], color=plt.cm.RdYlGn_r(df_sorted["loyer_m2"] / df_sorted["loyer_m2"].max()), zorder=2)
ax.errorbar(
    x, df_sorted["loyer_m2"],
    yerr=[df_sorted["loyer_m2"] - df_sorted["loyer_m2_bas"],
          df_sorted["loyer_m2_haut"] - df_sorted["loyer_m2"]],
    fmt="none", ecolor="black", capsize=3, linewidth=1, zorder=3
)
ax.set_xticks(x)
ax.set_xticklabels(df_sorted["arrondissement"].astype(str), rotation=45)
ax.set_xlabel("Arrondissement (trié par loyer décroissant)")
ax.set_ylabel("Loyer médian prédit (€/m²/mois)")
ax.set_title("Loyers d'annonce par arrondissement parisien — 2025")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:.0f} €/m²"))
plt.tight_layout()
plt.show()
print(f"\nLoyer médian Paris : {df['loyer_m2'].median():.1f} €/m²/mois")
print(f"Arrondissement le plus cher : {df.loc[df['loyer_m2'].idxmax(), 'libgeo']} — {df['loyer_m2'].max():.1f} €/m²")
print(f"Arrondissement le moins cher : {df.loc[df['loyer_m2'].idxmin(), 'libgeo']} — {df['loyer_m2'].min():.1f} €/m²")


In [ ]:
# ── Distribution des loyers ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df["loyer_m2"], bins=15, color="#2196F3", edgecolor="white", rwidth=0.9)
axes[0].axvline(df["loyer_m2"].mean(), color="red", linestyle="--",
                label=f"Moyenne : {df['loyer_m2'].mean():.1f} €/m²")
axes[0].set_title("Distribution des loyers médians")
axes[0].set_xlabel("€/m²/mois")
axes[0].legend()

# Intervalle de confiance moyen par arrondissement
df_sorted2 = df.sort_values("arrondissement")
ci_width = df_sorted2["loyer_m2_haut"] - df_sorted2["loyer_m2_bas"]
axes[1].bar(df_sorted2["arrondissement"].astype(str), ci_width,
            color=plt.cm.Blues(ci_width / ci_width.max()))
axes[1].set_xlabel("Arrondissement")
axes[1].set_ylabel("Largeur IC 95% (€/m²)")
axes[1].set_title("Incertitude de prédiction par arrondissement")
plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
# ── Corrélation loyers × nombre d'observations ───────────────────────────────
fig, ax = plt.subplots(figsize=(10, 6))
sc = ax.scatter(df["nb_obs"], df["loyer_m2"],
                c=df["arrondissement"], cmap="tab20", s=80, zorder=5)
for _, row in df.iterrows():
    ax.annotate(str(row["arrondissement"]),
                (row["nb_obs"], row["loyer_m2"]),
                xytext=(5, 3), textcoords="offset points", fontsize=8)
plt.colorbar(sc, ax=ax, label="Arrondissement")
ax.set_xlabel("Nombre d'observations")
ax.set_ylabel("Loyer médian prédit (€/m²/mois)")
ax.set_title("Loyer vs Volume d'annonces")
corr = df["nb_obs"].corr(df["loyer_m2"])
ax.text(0.05, 0.95, f"Corrélation : {corr:.3f}", transform=ax.transAxes, fontsize=11)
plt.tight_layout()
plt.show()


## Conclusions EDA
- Écart de loyers de ~2x entre les arrondissements les plus chers (6e, 7e, 8e) et les moins chers (19e, 20e).
- Les intervalles de confiance sont plus larges dans les arrondissements avec peu d'observations.
- Le loyer médian d'annonce complète le prix de vente DVF pour une vision locataire/propriétaire.
- Cet indicateur entre dans le Gold `arrondissement_summary` comme `median_price`.
